## Задача

Обучить RNN на каком-то текстовом датасете и генерировать новый текст с теми же паттернами, что и исходный.

## Реализация

### Подготовка и загрузка данных

In [44]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [45]:
import torch
import re

In [46]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cuda


In [47]:
# Сохраняем URL
gist_url = "https://gist.github.com/bdcb66640cc070450817686f6c818897.git"

In [48]:
!git clone {gist_url}

fatal: destination path 'bdcb66640cc070450817686f6c818897' already exists and is not an empty directory.


In [49]:
MAX_CHARS = 500000
with open('bdcb66640cc070450817686f6c818897//war_and_peace.ru.txt', 'r', encoding='utf-8') as file:
    dataset = file.read(MAX_CHARS)

In [50]:
BATCH_SIZE = 64
SEQ_LENGTH = 20 # длина входной последовательности каждого примера
VOCAB_SIZE = 10000  # ограничим размер словаря
EMBEDDING_SIZE = 128
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
LEARNING_RATE = 0.001
N_EPOCHS = 10

## Словарь и Датасет

In [51]:
def tokenize_text(text):
    # Простая токенизация на слова и знаки препинания
    # Берем слова и отдельные знаки препинания
    tokens = re.findall(r'\b\w+\b|[.,!?;:"\'()—\-]', text.lower())
    return tokens

In [52]:
# Токенизируем весь датасет
tokens = tokenize_text(dataset)
print(f"Всего токенов (слов и знаков препинания): {len(tokens)}")

Всего токенов (слов и знаков препинания): 96183


In [53]:
vocab = sorted(set(tokens))
vocab_size = len(vocab)

In [54]:
print(f"Размер словаря (все уникальные токены): {vocab_size}")

Размер словаря (все уникальные токены): 16852


In [55]:
# ИСПРАВЛЕНО: меняем названия на word вместо char
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for idx, word in enumerate(vocab)}

In [56]:
def text_to_indices(text_tokens):
    # Простое преобразование токенов в индексы
    indices = []
    for token in text_tokens:
        if token in word2idx:
            indices.append(word2idx[token])
        else:
            # Если токен не найден (маловероятно, но на всякий случай)
            continue
    return indices

In [57]:
# Преобразуем весь датасет в индексы
dataset_indices = text_to_indices(tokens)

print(f"Всего токенов в датасете после преобразования: {len(dataset_indices)}")

Всего токенов в датасете после преобразования: 96183


In [58]:
class WordDataset(torch.utils.data.Dataset):
    def __init__(self, data,seq_len):
        self.data = data  # data - список индексов
        self.seq_len = seq_len # seq_length - длина последовательности

    def __len__(self):
        return len(self.data) -self.seq_len

    def __getitem__(self, index):

         x = self.data[index:(index + self.seq_len)]

         y = self.data[index + 1:index + self.seq_len + 1]

         x_tensor = torch.tensor(x)
         y_tensor = torch.tensor(y)
         return x_tensor,y_tensor


Train и test

In [59]:
split_idx = int(len(dataset_indices) * 0.8)
train_data = dataset_indices[:split_idx]
test_data = dataset_indices[split_idx:]

train_dataset = WordDataset(train_data, SEQ_LENGTH)
test_dataset = WordDataset(test_data, SEQ_LENGTH)

In [60]:
BATCH_SIZE = 64

In [61]:
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

## Модель и обучение

In [62]:
class WordRNN(torch.nn.Module):
    def __init__(self,vocab_size_, embedding_size,hidden_size,num_layers):
        super().__init__()
        self.vocab_size = vocab_size_
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Первый слой
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size_,
            embedding_dim=embedding_size
        )

        # Второй слой
        self.rnn = torch.nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout = 0.2 if num_layers > 1 else 0  # dropout только между слоями
        )

        # Третий слой
        self.lin = torch.nn.Linear(
            in_features= hidden_size,
            out_features = vocab_size_
        )

        self.dropout = torch.nn.Dropout(0.2)

    def forward(self,x, hidden = None):

        embedded = self.embedding(x)

        # dropout к эмбеддингам
        embedded = self.dropout(embedded)

        output, hidden = self.rnn(embedded, hidden)

        output = output.contiguous().view(-1, self.hidden_size)

        # dropout к выходу RNN
        output = self.dropout(output)

        logits = self.lin(output)

        return logits,hidden


In [63]:
def train_epoch(model,dataloader,criterion,optimizer,device):
    model.train()
    clip_norm = 0.5
    total_loss =0
    total_samples =0

    for batch_idx, (inputs,targets) in enumerate(dataloader):

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        # НЕ передаем hidden - каждый батч обрабатывается независимо
        logits, _ = model(inputs)

        loss = criterion(logits,targets.view(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        if batch_idx % (batch_size*10*2) == 0:
            print(f'Batch {batch_idx}/{len(dataloader)} Loss: {total_loss / total_samples:.3f}')

    return total_loss / total_samples



проверка на test

In [64]:
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for batch_idx, (inputs,targets) in enumerate(dataloader):
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits,targets.view(-1))

            batch_size = inputs.size(0)
            total_loss += loss.item()*batch_size
            total_samples += batch_size

    return total_loss/total_samples

In [65]:
def train(model,train_loader,test_loader,n_epochs,device):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

    # Сохранение лучшей модели
    best_val_loss = None
    best_model = None

    print("Начинаем обучение...")
    for epoch in range(n_epochs):
        print(f'Epoch: {epoch+1}/{n_epochs}')
        train_loss = train_epoch(model,train_loader,criterion,optimizer,device)

        test_loss = validate(model,test_loader,criterion,device)

        if best_model is None:
            best_model=model.state_dict().copy()
            best_val_loss = test_loss

        if  test_loss < best_val_loss:
            best_model=model.state_dict().copy()
            best_val_loss = test_loss

        print(f"Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

        # Загружаем лучшую модель
        if best_model is not None:
            model.load_state_dict(best_model)
    return model

## Запуск модели

In [66]:
model = CharRNN(
    vocab_size_=vocab_size,
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
).to(device)

In [67]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    n_epochs=N_EPOCHS,
    device=device
)

Начинаем обучение...
Epoch: 1/10
Batch 0/1202 Loss: 9.733
Train Loss: 6.7401 | Test Loss: 7.6834
Epoch: 2/10
Batch 0/1202 Loss: 6.336
Train Loss: 5.9272 | Test Loss: 7.9154
Epoch: 3/10
Batch 0/1202 Loss: 5.492
Train Loss: 5.4367 | Test Loss: 8.1921
Epoch: 4/10
Batch 0/1202 Loss: 5.213
Train Loss: 5.0739 | Test Loss: 8.4664
Epoch: 5/10
Batch 0/1202 Loss: 4.948
Train Loss: 4.7987 | Test Loss: 8.7602
Epoch: 6/10
Batch 0/1202 Loss: 4.694
Train Loss: 4.5861 | Test Loss: 9.0411
Epoch: 7/10
Batch 0/1202 Loss: 4.447
Train Loss: 4.4207 | Test Loss: 9.3320
Epoch: 8/10
Batch 0/1202 Loss: 4.287
Train Loss: 4.2891 | Test Loss: 9.5994
Epoch: 9/10
Batch 0/1202 Loss: 4.191
Train Loss: 4.1830 | Test Loss: 9.8261
Epoch: 10/10
Batch 0/1202 Loss: 4.121
Train Loss: 4.0975 | Test Loss: 10.0840


CharRNN(
  (embedding): Embedding(16852, 128)
  (rnn): LSTM(128, 128, num_layers=2, dropout=0.2)
  (lin): Linear(in_features=128, out_features=16852, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

## Результат

In [68]:
def tokens_to_text(token_list):
    """Преобразует список токенов обратно в текст"""
    text = ""
    for i, token in enumerate(token_list):
        # Если токен - знак препинания, не добавляем пробел перед ним
        if i > 0 and token not in '.,!?;:"\'()—\-':
            # Если предыдущий токен не был знаком препинания, добавляем пробел
            if token_list[i-1] not in '.,!?;:"\'()—\-':
                text += " "
        text += token
    return text

<>:6: SyntaxWarning: invalid escape sequence '\-'
<>:8: SyntaxWarning: invalid escape sequence '\-'
<>:6: SyntaxWarning: invalid escape sequence '\-'
<>:8: SyntaxWarning: invalid escape sequence '\-'
C:\Users\alesh\AppData\Local\Temp\ipykernel_784\1735741714.py:6: SyntaxWarning: invalid escape sequence '\-'
  if i > 0 and token not in '.,!?;:"\'()—\-':
C:\Users\alesh\AppData\Local\Temp\ipykernel_784\1735741714.py:8: SyntaxWarning: invalid escape sequence '\-'
  if token_list[i-1] not in '.,!?;:"\'()—\-':


In [69]:
def generate_text(model, seed_text, length=50, temperature=1.0, device='cpu'):
    """Генерирует текст на уровне СЛОВ"""
    model.eval()

    # Токенизируем seed текст
    seed_tokens = tokenize_text(seed_text)

    # Преобразуем в индексы
    input_seq = [word2idx[token] for token in seed_tokens if token in word2idx]

    if not input_seq:
        # Если ни одного токена не найдено, начинаем с любого слова
        input_seq = [0]

    # Обрезаем, если слишком длинный
    if len(input_seq) > SEQ_LENGTH:
        input_seq = input_seq[-SEQ_LENGTH:]

    # Конвертируем в тензор [1, seq_len]
    current_input = torch.tensor(input_seq).unsqueeze(0).to(device)

    generated_tokens = []
    hidden = None

    for _ in range(length):
        with torch.no_grad():
            logits, hidden = model(current_input, hidden)

            # Берем последнее предсказание
            last_logits = logits[-1, :] / temperature

            # Применяем softmax
            probs = torch.nn.functional.softmax(last_logits, dim=-1)

            # Выбираем следующее слово
            next_token_idx = torch.multinomial(probs, 1).item()
            generated_tokens.append(idx2word[next_token_idx])

            # Обновляем вход
            next_token_tensor = torch.tensor([[next_token_idx]]).to(device)
            current_input = torch.cat([current_input[:, 1:], next_token_tensor], dim=1)

    # Объединяем seed и сгенерированные токены
    all_tokens = seed_tokens + generated_tokens

    # Преобразуем обратно в текст
    return tokens_to_text(all_tokens)


In [70]:
def test_generation(model, device):
    print("Тестируем генерацию текста на уровне СЛОВ...")
    print("=" * 50)

    # Несколько примеров для генерации
    seed_texts = [
        "князь",
        "война",
        "любовь",
        "пьер",
        "наполеон"
    ]

    for i, seed in enumerate(seed_texts):
        print(f"\nПример {i + 1}:")
        print(f"Seed: '{seed}'")
        print("-" * 30)

        # Генерируем с разной температурой
        for temp in [0.5, 0.8, 1.0, 1.2]:
            generated = generate_text(
                model=model,
                seed_text=seed,
                length=30,  # генерируем 30 слов (вместо 200 символов)
                temperature=temp,
                device=device
            )
            print(f"\nТемпература {temp}:")
            print(generated)
            print("-" * 30)


## Запуск модели - ИСПРАВЛЕНО название модели
model = WordRNN(  # ИЗМЕНИЛИ НА WordRNN
    vocab_size_=vocab_size,
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
).to(device)

In [71]:
test_generation(model, device)

Тестируем генерацию текста на уровне СЛОВ...

Пример 1:
Seed: 'князь'
------------------------------

Температура 0.5:
князь трофея интонации отлучаться беспорядок занимало пленный суеверие слетала vilaine умирающего павлоградский узнавая baron forte политические сначала взаимно ours взгляде ясно гуще желтоватые marieiages положению пособия стеснительна половину това строившихся мое
------------------------------

Температура 0.8:
князь злобно указывали косился запог прикушенною колонне оба passage печатания принять название покойно понять дружок соринки провианта пользовавшимся наклонясь снурок запомнить опасаясь поколебало последние дег рукавами вчерашнем прервал каждому кадушке поймет
------------------------------

Температура 1.0:
князь pitt ехать фамильярности забывая нужным providence венгерское любящих проезжая прислала осмелитесь забыты вниз искреннее избави невестка доказывать comprendre бегавшим австрийскую ко скверная потемкины заскрипел получать остроумное высокого брате т